# Service ML : factures (FastAPI + ngrok)

Run the cells in order. Before the first run: **Runtime → Restart session** (an old server may still be using port 8000).
Put your ngrok token in Colab **Secrets** (key icon on the left) under the name `NGROK_TOKEN`, with notebook access enabled.

In [1]:
!pip -q install fastapi uvicorn python-multipart nest_asyncio pyngrok
!apt-get -qq install tesseract-ocr tesseract-ocr-fra
!pip -q install pytesseract


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip
'apt-get' is not recognized as an internal or external command,
operable program or batch file.

[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Drive and models

In [2]:
import os, io, re, glob, time, threading
from datetime import datetime
import numpy as np, tensorflow as tf, pytesseract, uvicorn, nest_asyncio
from PIL import Image
from fastapi import FastAPI, UploadFile, File, HTTPException
from google.colab import drive

drive.mount('/content/drive')

def first_existing(candidates, name):
    for p in candidates:
        if os.path.exists(p):
            return p
    found = glob.glob('/content/drive/MyDrive/**/*.keras', recursive=True)
    raise FileNotFoundError(f"{name} not found. Tried {candidates}. .keras files on Drive: {found}")

TYPE_MODEL = first_existing([
    "/content/drive/MyDrive/invoice_classifier.keras",
    "/content/drive/MyDrive/DL_Project/invoice_classifier.keras",
], "Type model")
QUALITY_MODEL = first_existing([
    "/content/drive/MyDrive/DL_Project/quality_classifier.keras",
    "/content/drive/MyDrive/quality_classifier.keras",
], "Quality model")
print("type model   :", TYPE_MODEL)
print("quality model:", QUALITY_MODEL)

type_model = tf.keras.models.load_model(TYPE_MODEL)
quality_model = tf.keras.models.load_model(QUALITY_MODEL)
TYPE_CLASSES = ["invoice", "other", "receipt"]   # alphabetical folder order
QUALITY_CLASSES = ["good", "poor"]

ModuleNotFoundError: No module named 'google.colab'

## 2. Extraction rules (OCR text → fields)

In [ ]:
MONTH = r"(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*"
DATE_PATTERNS = [
    rf"\b\d{{1,2}}[-/ .]{MONTH}[-/ .,]*\d{{2,4}}\b",
    rf"\b{MONTH}[ .]\d{{1,2}},? \d{{4}}\b",
    r"\b\d{4}[/\-.]\d{1,2}[/\-.]\d{1,2}\b",
    r"\b\d{1,2}[/\-.]\d{1,2}[/\-.]\d{2,4}\b",
]

def find_date(text):
    for line in text.splitlines():
        if re.search(r"date", line, re.I) and not re.match(r"\s*due", line, re.I):
            for p in DATE_PATTERNS:
                m = re.search(p, line, re.I)
                if m:
                    return m.group(0)
    return None

def find_supplier(text):
    for line in text.splitlines():
        l = re.sub(r"(commercial|tax)?\s*invoice", "", line, flags=re.I).strip()
        if len(re.findall(r"[A-Za-z]", l)) < 3: continue
        if re.match(r"\W*A\w{1,3}ress", l, re.I): continue
        if re.match(r"\d", l): continue
        if re.search(r",\s*[A-Z]{2}\s*\d{4,5}", l): continue
        if re.search(r"description|unit price|quantity|amount", l, re.I): continue
        if re.search(r"\bid\b|inv/|date|bill|#", l, re.I): continue
        name = re.split(r"\s*\W?\s*A\w{1,3}ress", l, flags=re.I)[0].strip()
        if len(name) >= 3:
            return name
    return None

def find_total(text):
    for line in reversed(text.splitlines()):
        if re.search(r"total|montant|amount|ttc", line, re.I):
            nums = re.findall(r"\d+[.,]\d{2}", line)
            if nums:
                return float(nums[-1].replace(",", "."))
    return None

def iso_date(s):
    if not s:
        return None
    for fmt in ("%d-%b-%Y", "%d/%m/%Y", "%Y-%m-%d"):
        try:
            return datetime.strptime(s, fmt).strftime("%Y-%m-%d")
        except ValueError:
            pass
    return s

## 3. API (`POST /analyze`)

In [ ]:
def predict(model, classes, img):
    x = img.convert("RGB").resize((224, 224), Image.BILINEAR)
    x = np.expand_dims(np.array(x, dtype="float32"), 0)   # 0-255: normalisation is inside the model
    p = model.predict(x, verbose=0)[0]
    i = int(p.argmax())
    return classes[i], float(p[i])

app = FastAPI()

@app.post("/analyze")
async def analyze(file: UploadFile = File(...)):
    try:
        img = Image.open(io.BytesIO(await file.read()))
        img.load()
    except Exception:
        raise HTTPException(status_code=400, detail="Unsupported image format")

    dtype, dconf = predict(type_model, TYPE_CLASSES, img)
    qlabel, _ = predict(quality_model, QUALITY_CLASSES, img)

    warnings = []
    if qlabel == "poor":
        warnings.append("Image floue ou mal cadrée : reprendre la photo.")

    # no 'currency' key: the app's schema has a default value for it
    fields = {"supplier": None, "invoice_number": None, "invoice_date": None,
              "subtotal": None, "tax": None, "total": None}
    ocr_text = ""

    if dtype == "invoice":
        ocr_text = pytesseract.image_to_string(img.convert("L"), lang="fra+eng")
        total = find_total(ocr_text)
        d = iso_date(find_date(ocr_text))
        fields.update({
            "supplier": find_supplier(ocr_text),
            "invoice_date": d if d and re.fullmatch(r"\d{4}-\d{2}-\d{2}", d) else None,
            "total": f"{total:.2f}" if total is not None else None,
        })
    elif dtype == "receipt":
        warnings.append("Extraction des champs non prise en charge pour les tickets.")

    return {
        "mode": "real",
        "model_version": "mobilenetv2-v1",
        "classification": dtype,
        "confidence": round(dconf, 3),
        "quality": "acceptable" if qlabel == "good" else "retake_required",
        "warnings": warnings,
        "ocr_text": ocr_text,
        "fields": fields,
    }

## 4. Start the server

In [ ]:
import requests

nest_asyncio.apply()
threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000), daemon=True).start()

for _ in range(30):
    try:
        if requests.get("http://localhost:8000/docs", timeout=2).status_code == 200:
            print("Server is up")
            break
    except Exception:
        time.sleep(1)
else:
    print("Server did not start: restart the session and run all cells again")

## 5. Local test

In [ ]:
files = sorted(glob.glob("/content/drive/MyDrive/DL_Project/invoice_project_data/data/test/invoice/*"))
print(len(files), "test invoices found")
with open(files[0], "rb") as f:
    r = requests.post("http://localhost:8000/analyze", files={"file": (os.path.basename(files[0]), f, "image/png")})
print(r.status_code)
print(r.text[:1200])

## 6. Public URL (ngrok)

In [ ]:
from google.colab import userdata
from pyngrok import ngrok

ngrok.set_auth_token(userdata.get("3MFF9Pyzdt7mWJiIoCKJzjPBjAW_6oEjFDScgcUxRb55kqcXM"))
ngrok.kill()
tunnel = ngrok.connect(8000)
print("ML_URL =", tunnel.public_url + "/analyze")